In [ ]:
#install pypdf, scikit-learn, nltk

In [ ]:
import re
import nltk
import numpy as np
from PyPDF2 import PdfReader
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

nltk.download('punkt')

faq_pdf_path = "The_Bucket.fud_FAQ.pdf"

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\radu6\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!


In [3]:
# 1. Extract text from the PDF
def extract_faq_text(pdf_path):
    reader = PdfReader(pdf_path)
    text = ""
    for page in reader.pages:
        text += page.extract_text() + "\n"
    text = text.replace('\n', ' ').replace('  ', '')  # Replace newlines with spaces for easier processing
    return text

# 2. Split text into Q&A pairs (simple split by 'Q:' and 'A:')
def split_faq_sections(text):
    # This assumes the FAQ is formatted with 'Q:' and 'A:' markers
    qa_pairs = re.findall(r'(Q:.*?)(?=Q:|$)', text, re.DOTALL)
    questions = []
    answers = []
    for pair in qa_pairs:
        q_match = re.search(r'Q:(.*?)(A:)', pair, re.DOTALL)
        a_match = re.search(r'A:(.*)', pair, re.DOTALL)
        if q_match and a_match:
            questions.append(q_match.group(1).strip())
            answers.append(a_match.group(1).strip())
    return questions, answers

# 3. Vectorize the questions
def fit_vectorizer(questions):
    vectorizer = TfidfVectorizer(stop_words='english')
    question_vectors = vectorizer.fit_transform(questions)
    return vectorizer, question_vectors

# 4. Find the most relevant answer
def get_best_answer(user_question, vectorizer, question_vectors, answers):
    user_vec = vectorizer.transform([user_question])
    similarities = cosine_similarity(user_vec, question_vectors).flatten()
    best_idx = np.argmax(similarities)
    return answers[best_idx], similarities[best_idx]

# --- Example usage ---
# Extract and process FAQ
faq_text = extract_faq_text(faq_pdf_path)
questions, answers = split_faq_sections(faq_text)
vectorizer, question_vectors = fit_vectorizer(questions)

# Example: Get answer for a user question
user_question = "takeout home option"
best_answer, score = get_best_answer(user_question, vectorizer, question_vectors, answers)
# Remove trailing number (and optional period) from the answer
import re
cleaned_answer = re.sub(r'\s*\d+\.*$', '', best_answer)
print(f"Answer: {cleaned_answer}\n(Similarity: {score:.2f})")


Answer: We offer takeout services. You can place your order by phone. We partner with Foodpanda and Pathao Food for delivery.
(Similarity: 0.60)
